# 00H — Capability checks with three inputs

Fill in **bucket, role ARN and container image URI** in the first cell.
Everything else is chosen below. There are no enable switches.

Use a fresh kernel and run the cells in order. Every AWS operation remains visible. If a cell fails,
keep its original error and continue with the next independent service section.
Readback cells need the corresponding creation/write to have succeeded.

The notebook writes only synthetic data under a new
`capability-checks/<unique-name>/` folder in your bucket. It creates an offline
feature group, a tiny training job, a pending Registry package and a small
MLflow server. Resources remain for inspection; manual cleanup commands are
at the end and are not executed by Run All.

The supplied role is passed unchanged to Feature Store, Training and MLflow.
API calls still use the notebook's current credentials. No IAM role or policy
is created. A role rejection is a diagnostic result, not another input to find.

## 1. The only three values to fill in

- Enter a bucket name, or s3:// followed by that bucket name.
- Paste the exact IAM role ARN, including its existing path.
- Paste the ECR image URI. A repository address with no version on the end also works;
  the next cell then pins that repository's newest tagged image.

In [ ]:
bucket = ""
role_arn = ""
image_uri = ""


## 2. Choose test names and paths

- Use the notebook Region, or the Region in the ECR image URI if none is configured.
- Choose a unique resource name and a subfolder inside your bucket. Nothing is created or written here.
- If the image URI has no version on the end, look up and pin the newest tagged image in that repository.
- Create clients with short timeouts.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse
import io
import json
import re
import subprocess
import sys
import tarfile
import time
import uuid
import boto3
from botocore.config import Config

bucket = bucket.strip().removeprefix("s3://").rstrip("/")
role_arn, image_uri = role_arn.strip(), image_uri.strip()
assert bucket and "/" not in bucket, "Use just a bucket name, with or without s3://."
assert role_arn and image_uri, "Fill in the role ARN and image URI in cell 1."
region = boto3.Session().region_name
if not region:
    region = re.search(r"\.dkr\.ecr\.([^.]+)\.", image_uri).group(1)
session = boto3.Session(region_name=region)
client_config = Config(connect_timeout=5, read_timeout=20,
                       retries={"mode": "standard", "total_max_attempts": 1})
sm = session.client("sagemaker", config=client_config)
fs = session.client("sagemaker-featurestore-runtime", config=client_config)
s3 = session.client("s3", config=client_config)
glue = session.client("glue", config=client_config)
athena = session.client("athena", config=client_config)
if "@sha256:" not in image_uri and ":" not in image_uri.rsplit("/", 1)[-1]:
    # Repository address only: pin its newest tagged image, as 00G section 6 does.
    ecr = session.client("ecr", config=client_config)
    images = []
    for page in ecr.get_paginator("describe_images").paginate(
            registryId=image_uri.split(".", 1)[0], repositoryName=image_uri.split("/", 1)[1],
            filter={"tagStatus": "TAGGED"}):
        images.extend(page["imageDetails"])
    assert images, "That repository has no tagged images; paste a full image URI instead."
    newest = max(images, key=lambda item: item["imagePushedAt"])
    image_uri = image_uri + "@" + newest["imageDigest"]
    print("Pinned the newest tagged image:", newest.get("imageTags"))
name = "capability-" + uuid.uuid4().hex[:10]
prefix = "capability-checks/" + name
s3_root = f"s3://{bucket}/{prefix}"
feature_group_name = registry_group_name = tracking_server_name = training_job_name = name
record_id = "sample-1"
event_time = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
training_csv = b"x,y\n0,0\n1,2\n2,4\n3,6\n"
input_key = prefix + "/training-input/train.csv"
results = {}
package_arn = tracking_arn = mlflow_run_id = None
WORK = Path.cwd() / ".runs" / name
WORK.mkdir(parents=True, exist_ok=True)
print("Region:", region)
print("Role passed to services:", role_arn)
print("Image:", image_uri)
print("Will write test files under:", s3_root)
print("Will create resources named:", name)
print("Nothing has been created or written yet.")


## 3. Show the caller

- Call STS directly to show the notebook's current identity.
- Keep the supplied service role separate from this caller identity.
- A failure here does not remove the clients or generated paths.

In [ ]:
identity = session.client("sts", config=client_config).get_caller_identity()
print(json.dumps(identity, indent=2, default=str))
results["STS caller"] = identity["Arn"]


## 4. List a few keys in the bucket

- Call ListObjectsV2 for the supplied bucket.
- Print at most five keys without reading other objects.
- A list denial does not prevent trying the independent write and read cells.

In [ ]:
print(s3.list_objects_v2(Bucket=bucket, MaxKeys=5))
results["S3 ListObjectsV2"] = "confirmed"


## 5. Write the tiny training CSV

- Write four synthetic rows under this notebook's unique prefix.
- Use the bucket's default encryption settings.
- The training job below will read this exact object.

In [ ]:
print(s3.put_object(Bucket=bucket, Key=input_key, Body=training_csv, ContentType="text/csv"))
results["S3 PutObject"] = "confirmed"


## 6. Read the file back

- Call GetObject for the exact file just written.
- Check the returned bytes against the generated CSV.
- This is notebook access; the training job later checks service-role access.

In [ ]:
response = s3.get_object(Bucket=bucket, Key=input_key)
content = response["Body"].read()
assert content == training_csv
print(content.decode())
results["S3 GetObject"] = "confirmed"


## 7. List Feature Store groups

- Call ListFeatureGroups independently of the S3 checks.
- Print the first page of names and statuses.
- Creation does not depend on permission to list existing groups.

In [ ]:
response = sm.list_feature_groups(MaxResults=10)
print(json.dumps(response["FeatureGroupSummaries"], indent=2, default=str))
results["ListFeatureGroups"] = "confirmed"


## 8. Create an offline feature group

- Create three synthetic fields using the one supplied role.
- Write offline data below the generated S3 prefix; online storage is disabled.
- Use Feature Store's default Glue catalog/table creation.

In [ ]:
response = sm.create_feature_group(
    FeatureGroupName=feature_group_name,
    RecordIdentifierFeatureName="record_id", EventTimeFeatureName="event_time",
    FeatureDefinitions=[
        {"FeatureName": "record_id", "FeatureType": "String"},
        {"FeatureName": "event_time", "FeatureType": "String"},
        {"FeatureName": "value", "FeatureType": "Fractional"}],
    OnlineStoreConfig={"EnableOnlineStore": False},
    OfflineStoreConfig={
        "S3StorageConfig": {"S3Uri": s3_root + "/features"},
        "DisableGlueTableCreation": False},
    RoleArn=role_arn,
    Tags=[{"Key": "Diagnostic", "Value": name}])
print(response)
results["CreateFeatureGroup"] = "accepted; readiness checked next"


## 9. Wait for feature-group creation

- Describe the generated group for up to ten minutes.
- Print state changes and the original service failure details.
- A created group still needs ingestion and offline readback checks.

In [ ]:
deadline = time.monotonic() + 600
while True:
    feature = sm.describe_feature_group(FeatureGroupName=feature_group_name)
    status = feature["FeatureGroupStatus"]
    print(status, feature.get("OfflineStoreStatus"), flush=True)
    if status != "Creating" or time.monotonic() >= deadline:
        break
    time.sleep(15)
print(json.dumps(feature, indent=2, default=str))
assert status == "Created", feature.get("FailureReason", status)
results["Feature group ready"] = "confirmed"


## 10. Write a feature record

- Call PutRecord with one synthetic identifier, timestamp and value.
- Target the offline store explicitly.
- A successful write is separate from the later offline publication.

In [ ]:
print(fs.put_record(
    FeatureGroupName=feature_group_name,
    Record=[{"FeatureName": "record_id", "ValueAsString": record_id},
            {"FeatureName": "event_time", "ValueAsString": event_time},
            {"FeatureName": "value", "ValueAsString": "7.0"}],
    TargetStores=["OfflineStore"]))
results["PutRecord offline"] = "confirmed"


## 11. Find the offline data

- Read the actual offline location from DescribeFeatureGroup.
- Wait up to fifteen minutes for Feature Store to publish a Parquet file.
- No file yet means publication is pending; original API denials remain visible.

In [ ]:
feature = sm.describe_feature_group(FeatureGroupName=feature_group_name)
offline_uri = feature["OfflineStoreConfig"]["S3StorageConfig"]["ResolvedOutputS3Uri"]
offline = urlparse(offline_uri)
deadline = time.monotonic() + 900
while True:
    response = s3.list_objects_v2(
        Bucket=offline.netloc, Prefix=offline.path.lstrip("/"), MaxKeys=1000)
    parquet = [item["Key"] for item in response.get("Contents", []) if item["Key"].endswith(".parquet")]
    if parquet or time.monotonic() >= deadline:
        break
    print("Waiting for offline publication:", offline_uri, flush=True)
    time.sleep(20)
assert parquet, "No offline Parquet file observed yet; rerun this cell later."
print("Offline file:", parquet[0])
results["Offline S3 publication"] = "confirmed"


## 12. Read the offline file

- Read the first bytes of the published Parquet object.
- Check its Parquet signature.
- The Athena cells below verify the actual record value.

In [ ]:
response = s3.get_object(Bucket=offline.netloc, Key=parquet[0], Range="bytes=0-3")
assert response["Body"].read() == b"PAR1"
print("Offline Parquet is readable.")
results["Offline S3 GetObject"] = "confirmed"


## 13. Read the Glue table

- Use the catalog/database/table created for the feature group.
- Call GetTable without requiring a separately supplied database name.
- Glue or Lake Formation errors remain visible at this command.

In [ ]:
feature = sm.describe_feature_group(FeatureGroupName=feature_group_name)
catalog = feature["OfflineStoreConfig"]["DataCatalogConfig"]
response = glue.get_table(DatabaseName=catalog["Database"], Name=catalog["TableName"])
print(json.dumps(response["Table"], indent=2, default=str))
results["Glue GetTable"] = "confirmed"


## 14. Query the offline record

- Query only the generated synthetic record in its feature table.
- Use Athena's default workgroup and this notebook's S3 results folder.
- Submission does not count as successful query execution or result access.

In [ ]:
feature = sm.describe_feature_group(FeatureGroupName=feature_group_name)
catalog = feature["OfflineStoreConfig"]["DataCatalogConfig"]
query = (f'SELECT record_id, value FROM "{catalog["Database"]}"."{catalog["TableName"]}" '
         f"WHERE record_id = '{record_id}' AND NOT is_deleted ORDER BY event_time DESC LIMIT 1")
response = athena.start_query_execution(
    QueryString=query,
    ResultConfiguration={"OutputLocation": s3_root + "/athena/"},
    QueryExecutionContext={"Catalog": catalog["Catalog"], "Database": catalog["Database"]})
query_id = response["QueryExecutionId"]
print(query_id, query)
results["Athena StartQueryExecution"] = "accepted"


## 15. Wait for the query

- Describe the submitted query for up to five minutes.
- Print its status and any service failure reason.
- Read the actual values in the next cell after success.

In [ ]:
deadline = time.monotonic() + 300
while True:
    query_result = athena.get_query_execution(QueryExecutionId=query_id)["QueryExecution"]
    status = query_result["Status"]["State"]
    print(status, flush=True)
    if status in {"SUCCEEDED", "FAILED", "CANCELLED"} or time.monotonic() >= deadline:
        break
    time.sleep(5)
assert status == "SUCCEEDED", query_result["Status"]
results["Athena query completed"] = "confirmed"


## 16. Verify the offline record value

- Call GetQueryResults for the completed query.
- Require the generated identifier and numeric value.
- An empty result is not counted as a successful feature read.

In [ ]:
response = athena.get_query_results(QueryExecutionId=query_id)
print(json.dumps(response["ResultSet"], indent=2))
rows = response["ResultSet"]["Rows"][1:]
assert rows, "No record returned yet; repeat the query after offline publication."
assert rows[0]["Data"][0]["VarCharValue"] == record_id
assert float(rows[0]["Data"][1]["VarCharValue"]) == 7.0
results["Athena offline readback"] = "confirmed"


## 17. Upload the tiny training script

- Write the visible Python program under this notebook's S3 prefix.
- The program fits four CSV rows and saves learned coefficients as JSON.
- The next cell asks SageMaker to download this script alongside the data.

In [ ]:
training_code = """
import csv, json
from pathlib import Path
from sklearn.linear_model import LinearRegression
with open("/opt/ml/input/data/training/train.csv") as stream:
    rows = list(csv.DictReader(stream))
model = LinearRegression().fit([[float(r["x"])] for r in rows], [float(r["y"]) for r in rows])
prediction = float(model.predict([[4.0]])[0])
output = Path("/opt/ml/model")
output.mkdir(parents=True, exist_ok=True)
(output / "model.json").write_text(json.dumps({
    "coefficient": float(model.coef_[0]), "intercept": float(model.intercept_),
    "prediction_for_4": prediction}))
assert abs(prediction - 8.0) < 1e-8
print("CAPABILITY_TRAINING_OK", prediction, flush=True)
"""
print(s3.put_object(Bucket=bucket, Key=prefix + "/training-code/train.py",
                    Body=training_code.encode(), ContentType="text/x-python"))
results["Training source upload"] = "confirmed"


## 18. Train a tiny model in the supplied image

- Download the script and CSV into separate SageMaker input channels, then run the script.
- Use the supplied role and image, one ml.m5.large, and a five-minute runtime limit.
- SageMaker uploads the saved coefficients as model.tar.gz automatically.
- This image must contain Python and scikit-learn; package errors remain job failures.

In [ ]:
response = sm.create_training_job(
    TrainingJobName=training_job_name, RoleArn=role_arn,
    AlgorithmSpecification={
        "TrainingImage": image_uri, "TrainingInputMode": "File",
        "ContainerEntrypoint": ["python", "/opt/ml/input/data/code/train.py"]},
    InputDataConfig=[{"ChannelName": "training", "DataSource": {"S3DataSource": {
        "S3Uri": s3_root + "/training-input/", "S3DataType": "S3Prefix",
        "S3DataDistributionType": "FullyReplicated"}}},
        {"ChannelName": "code", "DataSource": {"S3DataSource": {
            "S3Uri": s3_root + "/training-code/", "S3DataType": "S3Prefix",
            "S3DataDistributionType": "FullyReplicated"}}}],
    OutputDataConfig={"S3OutputPath": s3_root + "/training-output"},
    ResourceConfig={"InstanceType": "ml.m5.large", "InstanceCount": 1, "VolumeSizeInGB": 30},
    StoppingCondition={"MaxRuntimeInSeconds": 300},
    EnableNetworkIsolation=True,
    Tags=[{"Key": "Diagnostic", "Value": name}])
print(response)
results["CreateTrainingJob"] = "accepted"


## 19. Wait for training

- Describe the training job for up to twenty minutes, including instance startup.
- Print status and the service's failure reason.
- Capture the actual checkpoint URI automatically after completion.

In [ ]:
deadline = time.monotonic() + 1200
while True:
    training = sm.describe_training_job(TrainingJobName=training_job_name)
    status = training["TrainingJobStatus"]
    print(status, training.get("SecondaryStatus"), flush=True)
    if status in {"Completed", "Failed", "Stopped"} or time.monotonic() >= deadline:
        break
    time.sleep(15)
assert status == "Completed", training.get("FailureReason", status)
model_data_uri = training["ModelArtifacts"]["S3ModelArtifacts"]
print("Checkpoint:", model_data_uri)
results["Training completed"] = "confirmed"


## 20. Read and verify the checkpoint

- Read the model artifact location returned by SageMaker.
- Inspect the saved JSON without extracting or executing archive files.
- Verify that the fitted model predicted eight for the input four.

In [ ]:
model_location = urlparse(model_data_uri)
response = s3.get_object(Bucket=model_location.netloc, Key=model_location.path.lstrip("/"))
with tarfile.open(fileobj=io.BytesIO(response["Body"].read()), mode="r:gz") as archive:
    trained_model = json.load(archive.extractfile("model.json"))
assert abs(trained_model["prediction_for_4"] - 8.0) < 1e-8
print(trained_model)
results["Training model output readback"] = "confirmed"


## 21. List Registry groups

- Call ListModelPackageGroups independently of Feature Store and training.
- Print the first page of groups.
- Group creation below does not depend on this discovery call succeeding.

In [ ]:
response = sm.list_model_package_groups(MaxResults=10)
print(json.dumps(response["ModelPackageGroupSummaryList"], indent=2, default=str))
results["ListModelPackageGroups"] = "confirmed"


## 22. Create the diagnostic Registry group

- Create a uniquely named group with a diagnostic tag.
- Keep this group separate from application model groups.
- No model is approved by this operation.

In [ ]:
response = sm.create_model_package_group(
    ModelPackageGroupName=registry_group_name,
    ModelPackageGroupDescription="Temporary capability check",
    Tags=[{"Key": "Diagnostic", "Value": name}])
print(response)
results["CreateModelPackageGroup"] = "confirmed"


## 23. Register the tiny trained model

- Use the supplied image and the checkpoint returned by the training job.
- Register the diagnostic version as PendingManualApproval.
- This checks Registry writes; the JSON model is not an application serving package.

In [ ]:
training = sm.describe_training_job(TrainingJobName=training_job_name)
assert training["TrainingJobStatus"] == "Completed", training.get("FailureReason", training["TrainingJobStatus"])
model_data_uri = training["ModelArtifacts"]["S3ModelArtifacts"]
response = sm.create_model_package(
    ModelPackageGroupName=registry_group_name, ModelApprovalStatus="PendingManualApproval",
    InferenceSpecification={
        "Containers": [{"Image": image_uri, "ModelDataUrl": model_data_uri}],
        "SupportedContentTypes": ["application/json"],
        "SupportedResponseMIMETypes": ["application/json"]},
    SkipModelValidation="None")
package_arn = response["ModelPackageArn"]
print(response)
results["CreateModelPackage"] = "accepted"


## 24. Inspect the registered version

- Describe the package created by the preceding cell.
- Print its status, approval state, image and model artifact URI.
- Leave the package pending; no automatic approval or inference is performed.

In [ ]:
package = sm.describe_model_package(ModelPackageName=package_arn)
print(json.dumps({key: package.get(key) for key in [
    "ModelPackageArn", "ModelPackageStatus", "ModelApprovalStatus", "InferenceSpecification"]},
    indent=2, default=str))
results["DescribeModelPackage"] = package["ModelPackageStatus"]


## 25. List tracking servers

- List MLflow tracking servers independently of the other services.
- Print their names and states.
- The next cell creates its own uniquely named server without needing an existing server name.

In [ ]:
response = sm.list_mlflow_tracking_servers()
print(json.dumps(response["TrackingServerSummaries"], indent=2, default=str))
results["ListMlflowTrackingServers"] = "confirmed"


## 26. Create a small MLflow server

- Use the same supplied role and a generated artifact folder in your bucket.
- Select MLflow 3.0.0 so the client cell can choose the matching version automatically.
- Disable automatic model registration; this test must not approve or register MLflow models.

In [ ]:
response = sm.create_mlflow_tracking_server(
    TrackingServerName=tracking_server_name, TrackingServerSize="Small",
    MlflowVersion="3.0.0", RoleArn=role_arn,
    ArtifactStoreUri=s3_root + "/mlflow", AutomaticModelRegistration=False,
    Tags=[{"Key": "Diagnostic", "Value": name}])
tracking_arn = response["TrackingServerArn"]
print(response)
results["CreateMlflowTrackingServer"] = "accepted"


## 27. Wait for the tracking server

- Describe the server for up to thirty minutes while it provisions.
- Print the exact server ARN, version and final state.
- Keep a service or network error visible rather than silently skipping the test.

In [ ]:
deadline = time.monotonic() + 1800
while True:
    tracking = sm.describe_mlflow_tracking_server(TrackingServerName=tracking_server_name)
    status = tracking["TrackingServerStatus"]
    print(status, tracking.get("IsActive"), flush=True)
    if status not in {"Creating", "Starting", "Updating"} or time.monotonic() >= deadline:
        break
    time.sleep(20)
print(json.dumps(tracking, indent=2, default=str))
assert status == "Created", tracking.get("ErrorMessage", status)
tracking_arn = tracking["TrackingServerArn"]
results["MLflow server ready"] = "confirmed"


## 28. Request a tracking UI session

- Call CreatePresignedMlflowTrackingServerUrl for the generated server name.
- Keep its temporary authorized URL in mlflow_ui_url rather than in printed output.
- This checks session creation; opening the UI is a separate browser check.

In [ ]:
response = sm.create_presigned_mlflow_tracking_server_url(
    TrackingServerName=tracking_server_name, ExpiresInSeconds=60, SessionExpirationDurationInSeconds=1800)
mlflow_ui_url = response["AuthorizedUrl"]
print("UI session URL is available in mlflow_ui_url.")
results["CreatePresignedMlflowTrackingServerUrl"] = "confirmed; browser not tested"


## 29. Install the matching MLflow client

- Install MLflow's lightweight client and the SageMaker authentication plugin in this kernel's environment.
- Use the environment's configured package index; no index URL or credentials are requested.
- Package-download errors are separate from AWS permission results.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "mlflow-skinny==3.0.0", "sagemaker-mlflow"],
    check=True)
import mlflow
print("MLflow client:", mlflow.__version__)


## 30. Create an experiment and run

- Connect to the server ARN returned by AWS.
- Create a diagnostic experiment and run using the notebook's credentials.
- This tests MLflow data-plane writes separately from server creation.

In [ ]:
mlflow.set_tracking_uri(tracking_arn)
mlflow_client = mlflow.tracking.MlflowClient()
experiment_id = mlflow_client.create_experiment(name)
run = mlflow_client.create_run(experiment_id, tags={"purpose": "capability-check"})
mlflow_run_id = run.info.run_id
print("Experiment:", experiment_id, "| run:", mlflow_run_id)
results["MLflow experiment/run"] = "confirmed"


## 31. Write and read a parameter and metric

- Write one parameter and one numeric metric.
- Read the run back and verify both values.
- Keep API errors visible in this cell.

In [ ]:
mlflow_client.log_param(mlflow_run_id, "source", "capability-check")
mlflow_client.log_metric(mlflow_run_id, "probe_value", 7.0)
run = mlflow_client.get_run(mlflow_run_id)
assert run.data.params["source"] == "capability-check"
assert run.data.metrics["probe_value"] == 7.0
print(run.data)
results["MLflow metric/parameter write/read"] = "confirmed"


## 32. Write and read an artifact

- Write one small local text file and upload it to the run.
- Download it to a separate directory and verify its bytes.
- Mark the run finished after successful artifact readback.

In [ ]:
artifact = WORK / "probe.txt"
artifact.write_text("artifact-access-confirmed\n")
mlflow_client.log_artifact(mlflow_run_id, str(artifact))
download_dir = WORK / "downloaded"
download_dir.mkdir(exist_ok=True)
downloaded = mlflow_client.download_artifacts(mlflow_run_id, "probe.txt", dst_path=str(download_dir))
assert Path(downloaded).read_bytes() == artifact.read_bytes()
mlflow_client.set_terminated(mlflow_run_id)
print("Artifact upload/download confirmed:", mlflow_run_id)
results["MLflow artifact write/read"] = "confirmed"


## 33. Capture the observed results

- Print successful operations recorded by their actual cells.
- Save resource names and results without the temporary UI URL.
- Missing operations remain unconfirmed; use the original exceptions beside their commands.

In [ ]:
print("CAPABILITY CHECK")
print("Region:", region)
print("Supplied role:", role_arn)
print("Image:", image_uri)
print("S3:", s3_root)
for operation, outcome in results.items():
    print(operation + ": " + outcome)
print("Unlisted operations are unconfirmed; keep their original cell errors.")
print("Not tested: online Feature Store, HPO, native Batch Transform, in-job MLflow connectivity.")
(WORK / "results.json").write_text(json.dumps({
    "region": region, "role_arn": role_arn, "image_uri": image_uri, "s3_root": s3_root,
    "resource_name": name, "results": results, "package_arn": package_arn,
    "tracking_arn": tracking_arn, "mlflow_run_id": mlflow_run_id}, indent=2) + "\n")
print("Local report:", WORK / "results.json")


## After inspection: manual cleanup

These are reference commands, **not executable notebook cells**, so Run All
retains the results. Run the relevant command later for a resource this notebook
successfully created. The unique names above identify only this diagnostic.

```python
sm.delete_feature_group(FeatureGroupName=feature_group_name)
sm.delete_model_package(ModelPackageName=package_arn)
sm.delete_model_package_group(ModelPackageGroupName=registry_group_name)
sm.delete_mlflow_tracking_server(TrackingServerName=tracking_server_name)
```

If training is still running, `sm.stop_training_job(TrainingJobName=training_job_name)`
stops this diagnostic job. Completed job history remains available.
Feature-group deletion retains its offline S3 data and Glue table. The exact
table is in `catalog`; `glue.delete_table(DatabaseName=catalog["Database"],
Name=catalog["TableName"])` removes it after inspection. The small S3 artifacts
remain only under the printed `s3_root`; remove that folder when no longer needed.

The default service settings deliberately remain visible in the API calls.
A requirement for a different role policy, VPC, KMS key, catalog or Athena
workgroup will appear as a service error; this notebook does not change account
policies or claim those errors are missing notebook inputs.